In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\| **Optimization** \|\| [Save &
Load Model](saveloadrun_tutorial.html)

学习基础知识 \|\| 快速入门 \|\| 张量 \|\| 数据集与数据加载器 \|\| 数据变换 \|\| 构建模型 \|\| 自动微分 \|\| **优化模型参数** \|\| 保存与加载模型

Optimizing Model Parameters 优化模型参数
===========================

Now that we have a model and data it\'s time to train, validate and test
our model by optimizing its parameters on our data. Training a model is
an iterative process; in each iteration the model makes a guess about
the output, calculates the error in its guess (*loss*), collects the
derivatives of the error with respect to its parameters (as we saw in
the [previous section](autogradqs_tutorial.html)), and **optimizes**
these parameters using gradient descent. For a more detailed walkthrough
of this process, check out this video on [backpropagation from
3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8).

现在我们有了模型和数据，是时候通过在数据上优化模型参数来训练、验证和测试模型了。训练模型是一个迭代过程：在每次迭代中，模型对输出做出猜测，计算猜测的误差（*损失*），收集误差关于各参数的导数（就像我们在[上一节](autogradqs_tutorial.html)中看到的那样），并使用梯度下降**优化**这些参数。如果想更详细地了解这个过程，可以看看 [3Blue1Brown 讲反向传播的视频](https://www.youtube.com/watch?v=tIeHLnjs5U8)。

Prerequisite Code 前置代码
-----------------

We load the code from the previous sections on [Datasets &
DataLoaders](data_tutorial.html) and [Build
Model](buildmodel_tutorial.html).

我们加载前面 [数据集与数据加载器](data_tutorial.html) 和 [构建模型](buildmodel_tutorial.html) 两节中的代码。


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

Hyperparameters 超参数
===============

Hyperparameters are adjustable parameters that let you control the model
optimization process. Different hyperparameter values can impact model
training and convergence rates ([read
more](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)
about hyperparameter tuning)

超参数是可调节的参数，用来控制模型的优化过程。不同的超参数值会影响模型的训练和收敛速度（[进一步了解](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)超参数调优）。

We define the following hyperparameters for training:

我们为训练定义以下超参数：

-   **Number of Epochs** - the number of times to iterate over the
    dataset
-   **Batch Size** - the number of data samples propagated through the
    network before the parameters are updated
-   **Learning Rate** - how much to update models parameters at each
    batch/epoch. Smaller values yield slow learning speed, while large
    values may result in unpredictable behavior during training.

*   **轮数（Number of Epochs）**：遍历数据集的次数。
*   **批大小（Batch Size）**：在更新参数之前，通过网络传播的数据样本数量。
*   **学习率（Learning Rate）**：每个批次/轮次中更新模型参数的幅度。值越小，学习速度越慢；值太大，则可能导致训练过程中出现不可预测的行为。


In [ ]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

Optimization Loop 优化循环
=================

Once we set our hyperparameters, we can then train and optimize our
model with an optimization loop. Each iteration of the optimization loop
is called an **epoch**.

设置好超参数后，就可以用优化循环来训练和优化模型了。优化循环的每一次迭代称为一个 **epoch**（轮次）。

Each epoch consists of two main parts:

每个 epoch 包括两个主要部分：

-   **The Train Loop** - iterate over the training dataset and try to
    converge to optimal parameters.
-   **The Validation/Test Loop** - iterate over the test dataset to
    check if model performance is improving.

*   **训练循环**：遍历训练数据集，尝试收敛到最优参数。
*   **验证/测试循环**：遍历测试数据集，检查模型性能是否在提升。

Let\'s briefly familiarize ourselves with some of the concepts used in
the training loop. Jump ahead to see the
`full-impl-label`{.interpreted-text role="ref"} of the optimization
loop.

我们先简单熟悉一下训练循环中用到的一些概念。也可以直接跳到下文的「完整实现」一节，查看优化循环的完整代码。

Loss Function 损失函数
-------------

When presented with some training data, our untrained network is likely
not to give the correct answer. **Loss function** measures the degree of
dissimilarity of obtained result to the target value, and it is the loss
function that we want to minimize during training. To calculate the loss
we make a prediction using the inputs of our given data sample and
compare it against the true data label value.

面对一些训练数据时，未经训练的网络很可能给不出正确答案。**损失函数**衡量得到的结果与目标值之间的差异程度，训练时我们要最小化的正是损失函数。为了计算损失，我们用给定数据样本的输入做出预测，并将其与真实的数据标签值进行比较。

Common loss functions include
[nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)
(Mean Square Error) for regression tasks, and
[nn.NLLLoss](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss)
(Negative Log Likelihood) for classification.
[nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss)
combines `nn.LogSoftmax` and `nn.NLLLoss`.

常见的损失函数包括用于回归任务的 [nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)（均方误差）和用于分类任务的 [nn.NLLLoss](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss)（负对数似然）。[nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss) 结合了 `nn.LogSoftmax` 和 `nn.NLLLoss`。

We pass our model\'s output logits to `nn.CrossEntropyLoss`, which will
normalize the logits and compute the prediction error.

我们把模型输出的 logits 传给 `nn.CrossEntropyLoss`，它会对 logits 进行归一化并计算预测误差。


In [ ]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

Optimizer 优化器
=========

Optimization is the process of adjusting model parameters to reduce
model error in each training step. **Optimization algorithms** define
how this process is performed (in this example we use Stochastic
Gradient Descent). All optimization logic is encapsulated in the
`optimizer` object. Here, we use the SGD optimizer; additionally, there
are many [different
optimizers](https://pytorch.org/docs/stable/optim.html) available in
PyTorch such as ADAM and RMSProp, that work better for different kinds
of models and data.

优化是在每个训练步骤中调整模型参数以减小模型误差的过程。**优化算法**定义了这个过程如何进行（本例中我们使用随机梯度下降）。所有优化逻辑都封装在 `optimizer` 对象中。这里我们使用 SGD 优化器；此外，PyTorch 中还有许多[不同的优化器](https://pytorch.org/docs/stable/optim.html)，比如 ADAM 和 RMSProp，它们分别更适合不同类型的模型和数据。

We initialize the optimizer by registering the model\'s parameters that
need to be trained, and passing in the learning rate hyperparameter.

我们通过注册需要训练的模型参数并传入学习率超参数来初始化优化器。


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

Inside the training loop, optimization happens in three steps:

在训练循环中，优化分三步进行：

-   Call `optimizer.zero_grad()` to reset the gradients of model
    parameters. Gradients by default add up; to prevent double-counting,
    we explicitly zero them at each iteration.
-   Backpropagate the prediction loss with a call to `loss.backward()`.
    PyTorch deposits the gradients of the loss w.r.t. each parameter.
-   Once we have our gradients, we call `optimizer.step()` to adjust the
    parameters by the gradients collected in the backward pass.

*   调用 `optimizer.zero_grad()` 重置模型参数的梯度。梯度默认会累加；为了防止重复计算，我们在每次迭代时显式地把梯度清零。
*   调用 `loss.backward()` 对预测损失进行反向传播。PyTorch 会把损失关于每个参数的梯度存起来。
*   得到梯度后，调用 `optimizer.step()`，用反向传播中收集到的梯度来调整参数。


Full Implementation 完整实现 {#full-impl-label}
===================

We define `train_loop` that loops over our optimization code, and
`test_loop` that evaluates the model\'s performance against our test
data.

我们定义 `train_loop` 来循环执行优化代码，并定义 `test_loop` 在测试数据上评估模型的性能。


In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

We initialize the loss function and optimizer, and pass it to
`train_loop` and `test_loop`. Feel free to increase the number of epochs
to track the model\'s improving performance.

我们初始化损失函数和优化器，并把它们传给 `train_loop` 和 `test_loop`。可以随意增加 epoch 数，观察模型性能的提升。


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Further Reading 延伸阅读
===============

-   [Loss
    Functions](https://pytorch.org/docs/stable/nn.html#loss-functions)
-   [torch.optim](https://pytorch.org/docs/stable/optim.html)
-   [Warmstart Training a
    Model](https://pytorch.org/tutorials/recipes/recipes/warmstarting_model_using_parameters_from_a_different_model.html)

*   [损失函数](https://pytorch.org/docs/stable/nn.html#loss-functions)
*   [torch.optim](https://pytorch.org/docs/stable/optim.html)
*   [热启动训练模型](https://pytorch.org/tutorials/recipes/recipes/warmstarting_model_using_parameters_from_a_different_model.html)
